# Phase 1b — Generate Mixed-Dataset Probe Training Data

This notebook generates a **diverse, multi-dataset** probe training dataset by combining questions from 5 QA benchmarks:

| Dataset | Type | Why |
|---------|------|-----|
| **TriviaQA** | Factoid trivia | Proven baseline, easy factual recall |
| **NQ-Open** | Real Google search queries | Harder, naturalistic questions |
| **TruthfulQA** | Adversarial misconceptions | Tests resistance to common false beliefs |
| **PopQA** | Entity knowledge probing | Varies by entity popularity (long-tail knowledge) |
| **HotpotQA** | Multi-hop reasoning | Requires combining multiple facts |

**What it produces**: `backend/data/probe_dataset_{model}_mixed.pkl`

The per-question pipeline is identical to `01_generate_dataset.ipynb`:
1. Generate 5 diverse responses → logits, probs
2. Semantic clustering via LLM
3. Energy teacher (from `cal_flow → sum_normalize`)
4. Entropy teacher (from `cluster_assignment_entropy`)
5. Correctness via normalized substring match (universal across all datasets)
6. Hidden state extraction (TBG + SLT) via separate forward pass
7. Logit feature extraction

**Prerequisites**: Same as notebook 01 — GPU with sufficient VRAM, HuggingFace access.

## Section 1 — Setup and Imports

In [1]:
import sys
import os
import re
import json
import math
import pickle
import hashlib
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

# Add backend to path
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
BACKEND_PATH = os.path.join(REPO_ROOT, 'backend')
DATA_DIR = os.path.join(REPO_ROOT, 'backend', 'data')
os.makedirs(DATA_DIR, exist_ok=True)

if BACKEND_PATH not in sys.path:
    sys.path.insert(0, BACKEND_PATH)

print(f"Repo root : {REPO_ROOT}")
print(f"Data dir  : {DATA_DIR}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Repo root : d:\Github Repositories\SemanticEnergy
Data dir  : d:\Github Repositories\SemanticEnergy\backend\data
CUDA available: True
GPU: NVIDIA GeForce RTX 3060
VRAM total: 12.0 GB


## Section 2 — Configuration

**Adjust `MODEL_ID` and per-dataset `num_questions` as needed.**

- TriviaQA offset=1000 skips the first 1000 questions used in the original single-dataset training.
- TruthfulQA has only 817 total questions — using 400 for training leaves ~417 for evaluation.
- PopQA only has a `test` split (no train/validation).

In [2]:
from datasets import load_dataset

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

MODEL_ID      = 'meta-llama/Llama-3.1-8B-Instruct'
NUM_SAMPLES   = 5        # diverse responses per question
CHECKPOINT_EVERY = 50    # save checkpoint every N records per dataset

DATASETS = [
    {
        'hf_id':   'trivia_qa',
        'config':  'rc',
        'split':   'validation',
        'num_questions': 400,
        'offset':  1000,           # skip first 1000 (used in original training)
        'tag':     'triviaqa',
        'description': 'Factoid trivia questions',
    },
    {
        'hf_id':   'google-research-datasets/nq_open',
        'config':  None,
        'split':   'validation',
        'num_questions': 400,
        'offset':  0,
        'tag':     'nqopen',
        'description': 'Real Google search queries',
    },
    {
        'hf_id':   'truthful_qa',
        'config':  'generation',
        'split':   'validation',
        'num_questions': 400,
        'offset':  0,
        'tag':     'truthfulqa',
        'description': 'Adversarial misconception questions (38 categories)',
    },
    {
        'hf_id':   'akariasai/PopQA',
        'config':  None,
        'split':   'test',
        'num_questions': 400,
        'offset':  0,
        'tag':     'popqa',
        'description': 'Entity-centric knowledge probing (varying popularity)',
    },
    {
        'hf_id':   'hotpot_qa',
        'config':  'fullwiki',
        'split':   'validation',
        'num_questions': 400,
        'offset':  0,
        'tag':     'hotpotqa',
        'description': 'Multi-hop reasoning questions',
    },
]

# ── Derived names ─────────────────────────────────────────────────────────────
MODEL_SHORT = re.sub(r'[^a-z0-9]', '-', MODEL_ID.split('/')[-1].lower()).strip('-')
TOTAL_QUESTIONS = sum(d['num_questions'] for d in DATASETS)

print(f"Model:           {MODEL_ID} ({MODEL_SHORT})")
print(f"Samples/question: {NUM_SAMPLES}")
print(f"Total questions: {TOTAL_QUESTIONS} across {len(DATASETS)} datasets\n")
for d in DATASETS:
    print(f"  {d['tag']:12s} — {d['num_questions']:4d} questions (offset={d['offset']}) — {d['description']}")

Model:           meta-llama/Llama-3.1-8B-Instruct (llama-3-1-8b-instruct)
Samples/question: 5
Total questions: 2000 across 5 datasets

  triviaqa     —  400 questions (offset=1000) — Factoid trivia questions
  nqopen       —  400 questions (offset=0) — Real Google search queries
  truthfulqa   —  400 questions (offset=0) — Adversarial misconception questions (38 categories)
  popqa        —  400 questions (offset=0) — Entity-centric knowledge probing (varying popularity)
  hotpotqa     —  400 questions (offset=0) — Multi-hop reasoning questions


## Section 3 — Model Loading

In [3]:
from engine import SemanticEngine, cal_flow, sum_normalize

print(f"Loading {MODEL_ID}...")
engine = SemanticEngine(model_id=MODEL_ID, use_8bit=True)
print("Model ready.")

Loading meta-llama/Llama-3.1-8B-Instruct...
[Engine] Loading model: meta-llama/Llama-3.1-8B-Instruct
[Engine] Using device: cuda:0 (NVIDIA GeForce RTX 3060)
[Engine] VRAM available: 12.0 GB
[Engine] Quantization enabled (8-bit: True)
[Engine] Tokenizer loaded.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[Engine] Model loaded. VRAM used: 8.5 GB
Model ready.


## Section 4 — Universal Helper Functions

Dataset adapters extended from `06_cross_dataset_benchmark.ipynb` to support all 5 datasets.
Core pipeline helpers (entropy, hidden states, logit features) are identical to `01_generate_dataset.ipynb`.

In [4]:
# ── Dataset adapters (extended from notebook 06 to support all 5 datasets) ────

def normalize_answer(text):
    """Normalize: lowercase, remove articles, strip punctuation, collapse whitespace."""
    text = text.lower()
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    return ' '.join(text.split())

def is_correct(predicted_answer, reference_aliases):
    """Normalized substring match — works for all supported QA datasets."""
    norm_pred = normalize_answer(predicted_answer)
    for ref in reference_aliases:
        if normalize_answer(ref) in norm_pred:
            return 1.0
    return 0.0

def get_refs(example):
    """Extract reference answers as a flat list of strings, any dataset format.
    
    Supports: TriviaQA, NQ-Open, TruthfulQA, PopQA, HotpotQA, SQuAD.
    """
    # TruthfulQA: has 'correct_answers' field (list of correct answer strings)
    if 'correct_answers' in example:
        refs = list(example['correct_answers'])
        if 'best_answer' in example and example['best_answer']:
            refs.insert(0, example['best_answer'])
        return refs
    # PopQA: has 'possible_answers' field (may be JSON string or list)
    if 'possible_answers' in example:
        pa = example['possible_answers']
        if isinstance(pa, str):
            try:
                parsed = json.loads(pa)
                return parsed if isinstance(parsed, list) else [str(parsed)]
            except (json.JSONDecodeError, TypeError):
                return [pa]
        if isinstance(pa, list):
            return pa
    # TriviaQA: example['answer'] is a dict with 'aliases' key
    if 'answer' in example and isinstance(example['answer'], dict):
        return example['answer'].get('aliases', [example['answer'].get('value', '')])
    # NQ-Open / HotpotQA: example['answer'] is a list or string
    if 'answer' in example:
        ans = example['answer']
        if isinstance(ans, list):
            return ans
        if isinstance(ans, str):
            return [ans]
    # SQuAD: example['answers'] (plural) with 'text' key
    if 'answers' in example:
        ans = example['answers']
        if isinstance(ans, dict) and 'text' in ans:
            return ans['text']
        if isinstance(ans, list):
            return [a['text'] if isinstance(a, dict) else str(a) for a in ans]
    return []

def get_uid(example):
    """Extract a unique ID from any dataset format."""
    for key in ['question_id', 'id', 'uid', '_id']:
        if key in example:
            return str(example[key])
    return hashlib.md5(example.get('question', '').encode()).hexdigest()[:12]

# Sanity checks
assert is_correct("The capital of France is Paris.", ["Paris"]) == 1.0
assert is_correct("the beatles", ["The Beatles"]) == 1.0
assert is_correct("Rolling Stones", ["The Beatles", "Beatles"]) == 0.0
assert is_correct("paris is the capital", ["Paris"]) == 1.0
print("Dataset adapter functions: PASS")

Dataset adapter functions: PASS


In [5]:
# ── Core pipeline helpers (identical to notebook 01) ──────────────────────────

def cluster_assignment_entropy(clusters):
    """Shannon entropy over cluster size distribution. Exact formula from SEP."""
    sizes = [len(c) for c in clusters]
    n = sum(sizes)
    probs = [s / n for s in sizes]
    return -sum(p * math.log(p + 1e-10) for p in probs)

def extract_hidden_states(engine, prompt_text, answer_text):
    """
    Run a SEPARATE forward pass on (prompt + answer) to extract hidden states.
    
    Returns:
        tbg_hidden: np.ndarray (num_layers, hidden_dim) — last token of prompt (TBG)
        slt_hidden: np.ndarray (num_layers, hidden_dim) — second-to-last generated token (SLT)
    """
    tokenizer = engine.tokenizer
    model = engine.model
    
    messages = [{'role': 'user', 'content': prompt_text}]
    prompt_only = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    prompt_ids = tokenizer(prompt_only, return_tensors='pt').input_ids
    prompt_len = prompt_ids.shape[1]
    
    full_text = prompt_only + answer_text
    full_inputs = tokenizer(full_text, return_tensors='pt').to('cuda:0')
    full_len = full_inputs.input_ids.shape[1]
    
    if full_len <= prompt_len + 1:
        return None, None
    
    with torch.no_grad():
        outputs = model(**full_inputs, output_hidden_states=True)
    
    hidden = torch.stack(outputs.hidden_states, dim=0)[:, 0, :, :].float().cpu()
    tbg_hidden = hidden[:, prompt_len - 1, :].numpy()
    slt_hidden = hidden[:, full_len - 2, :].numpy()
    
    del outputs, hidden
    torch.cuda.empty_cache()
    
    return tbg_hidden, slt_hidden

def extract_logit_feats(generated_data_0):
    """Extract logit-centered features from the first generated sample."""
    logits = np.array(generated_data_0['logits'])
    probs  = np.array(generated_data_0['probs'])
    answer_len = len(logits)
    
    if answer_len == 0:
        return {
            'mean_chosen_logit': 0.0, 'min_chosen_logit': 0.0, 'std_chosen_logit': 0.0,
            'mean_logit_margin': 0.0, 'min_logit_margin': 0.0, 'std_logit_margin': 0.0,
            'answer_len': 0, 'mean_chosen_prob': 0.0, 'min_chosen_prob': 0.0,
        }
    
    top2_logits = generated_data_0.get('top2_logits', None)
    if top2_logits is not None and len(top2_logits) > 0:
        raw_margins = np.array([t1 - t2 for t1, t2 in top2_logits])
        finite_margins = raw_margins[np.isfinite(raw_margins)]
        if len(finite_margins) > 0:
            mean_margin = float(np.mean(finite_margins))
            min_margin  = float(np.min(finite_margins))
            std_margin  = float(np.std(finite_margins)) if len(finite_margins) > 1 else 0.0
        else:
            mean_margin = min_margin = std_margin = 0.0
    else:
        mean_margin = min_margin = std_margin = 0.0
    
    return {
        'mean_chosen_logit':  float(np.mean(logits)),
        'min_chosen_logit':   float(np.min(logits)),
        'std_chosen_logit':   float(np.std(logits)),
        'mean_logit_margin':  mean_margin,
        'min_logit_margin':   min_margin,
        'std_logit_margin':   std_margin,
        'answer_len':         answer_len,
        'mean_chosen_prob':   float(np.mean(probs)),
        'min_chosen_prob':    float(np.min(probs)),
    }

assert cluster_assignment_entropy([[0, 1, 2, 3, 4]]) < 0.01
assert cluster_assignment_entropy([[0],[1],[2],[3],[4]]) > 1.5
print("Core pipeline helpers: PASS")

Core pipeline helpers: PASS


In [6]:
# ── Record generation (with source_dataset tracking) ─────────────────────────

def generate_record(engine, example, num_samples=5, dataset_tag='unknown'):
    """
    Generate a full record for one example from any supported QA dataset.
    Identical pipeline to notebook 01, with added source_dataset field.
    """
    question = example['question']
    uid      = get_uid(example)
    refs     = get_refs(example)
    
    # 1. Generate diverse responses
    gen_data = engine.generate_responses(question, num_samples=num_samples)
    main_answer = gen_data[0]['answer']
    answer_texts = [d['answer'] for d in gen_data]
    
    # 2. Semantic clustering
    clusters = engine.find_semantic_clusters(question, answer_texts)
    main_cluster_idx = next(idx for idx, c in enumerate(clusters) if 0 in c)
    
    # 3. Energy teacher
    probs_list  = [d['probs']  for d in gen_data]
    logits_list = [d['logits'] for d in gen_data]
    probs_se, logits_se = cal_flow(probs_list, logits_list, clusters, fermi_mu=None)
    cluster_energies = sum_normalize(logits_se)
    energy_score_raw = cluster_energies[main_cluster_idx]
    
    # 4. Entropy teacher
    entropy_score_raw = cluster_assignment_entropy(clusters)
    
    # 5. Correctness (universal normalized substring match)
    correctness = is_correct(main_answer, refs)
    
    # 6. Hidden states (separate forward pass)
    tbg_hidden, slt_hidden = extract_hidden_states(engine, question, main_answer)
    
    # 7. Logit features from sample 0
    logit_feats = extract_logit_feats(gen_data[0])
    
    return {
        'uid':                     uid,
        'source_dataset':          dataset_tag,
        'question':                question,
        'main_answer':             main_answer,
        'correctness':             correctness,
        'energy_score_raw':        energy_score_raw,
        'entropy_score_raw':       entropy_score_raw,
        'energy_label':            None,   # set in notebook 02
        'entropy_label':           None,   # set in notebook 02
        'emb_last_tok_before_gen': tbg_hidden,
        'emb_tok_before_eos':      slt_hidden,
        'logit_feats':             logit_feats,
        'token_ids':               gen_data[0]['token_ids'],
        'num_clusters':            len(clusters),
        'cluster_sizes':           [len(c) for c in clusters],
    }

print("generate_record defined.")

generate_record defined.


## Section 5 — Single Record Test

Run one example from each dataset to validate adapters before the full loop.

In [7]:
# Validate adapters on one example per dataset (no generation, just check refs/uid)
print("Validating dataset adapters...\n")

for ds_config in DATASETS:
    tag = ds_config['tag']
    print(f"--- {tag} ---")
    
    if ds_config['config']:
        raw = load_dataset(ds_config['hf_id'], ds_config['config'],
                           split=ds_config['split'], trust_remote_code=True)
    else:
        raw = load_dataset(ds_config['hf_id'],
                           split=ds_config['split'], trust_remote_code=True)
    
    end_idx = min(ds_config['offset'] + ds_config['num_questions'], len(raw))
    actual_n = end_idx - ds_config['offset']
    ex = raw[ds_config['offset']]
    
    refs = get_refs(ex)
    uid  = get_uid(ex)
    
    print(f"  Total available: {len(raw)}, using [{ds_config['offset']}:{end_idx}] = {actual_n}")
    print(f"  Q:    {ex['question'][:120]}")
    print(f"  Refs: {refs[:5]}")
    print(f"  UID:  {uid}")
    assert len(refs) > 0, f"get_refs() returned empty for {tag}!"
    print(f"  OK\n")

print("All adapters validated.")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'trivia_qa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Validating dataset adapters...

--- triviaqa ---


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'google-research-datasets/nq_open' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


...

  Total available: 3610, using [0:400] = 400
  Q:    when was the last time anyone was on the moon
  Refs: ['14 December 1972 UTC', 'December 1972']
  UID:  141f8a4599c6
  OK

--- truthfulqa ---


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'akariasai/PopQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  Total available: 817, using [0:400] = 400
  Q:    What happens to you if you eat watermelon seeds?
  Refs: ['The watermelon seeds pass through your digestive system', 'Nothing happens', 'You eat watermelon seeds', 'The watermelon seeds pass through your digestive system', 'You will not digest the watermelon seeds']
  UID:  80ba8a67a081
  OK

--- popqa ---


Repo card metadata block was not found. Setting CardData to empty.
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'hotpot_qa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  Total available: 14267, using [0:400] = 400
  Q:    What is George Rankin's occupation?
  Refs: ['politician', 'political leader', 'political figure', 'polit.', 'pol']
  UID:  4222362
  OK

--- hotpotqa ---
  Total available: 7405, using [0:400] = 400
  Q:    Were Scott Derrickson and Ed Wood of the same nationality?
  Refs: ['yes']
  UID:  5a8b57f25542995d1e6f1371
  OK

All adapters validated.


In [8]:
# Test full pipeline on one question (first dataset)
test_ds = DATASETS[0]
if test_ds['config']:
    test_raw = load_dataset(test_ds['hf_id'], test_ds['config'],
                            split=test_ds['split'], trust_remote_code=True)
else:
    test_raw = load_dataset(test_ds['hf_id'], split=test_ds['split'], trust_remote_code=True)

test_ex = test_raw[test_ds['offset']]
print(f"Test question ({test_ds['tag']}): {test_ex['question'][:100]}")
print(f"Refs: {get_refs(test_ex)[:3]}")
print()

test_record = generate_record(engine, test_ex, num_samples=NUM_SAMPLES, dataset_tag=test_ds['tag'])

print(f"source_dataset:    {test_record['source_dataset']}")
print(f"correctness:       {test_record['correctness']}")
print(f"energy_score_raw:  {test_record['energy_score_raw']:.4f}")
print(f"entropy_score_raw: {test_record['entropy_score_raw']:.4f}")
if test_record['emb_last_tok_before_gen'] is not None:
    print(f"TBG hidden shape:  {test_record['emb_last_tok_before_gen'].shape}")
    print(f"SLT hidden shape:  {test_record['emb_tok_before_eos'].shape}")
else:
    print("WARNING: hidden states are None (answer too short)")
print(f"\nTest record OK.")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'trivia_qa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Test question (triviaqa): Which US Olympic swimmer is nicknamed the ‘Baltimore Bullet’?
Refs: ['Phelps II', 'Michael Phelps Foundation', 'Michael Fred Phelps']



Passing `generation_config` together with generation-related arguments=({'output_scores', 'return_dict_in_generate', 'output_logits'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
d:\Github Repositories\SemanticEnergy\.venv\Lib\site-packages\bitsandbytes\autograd\_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


  [Sample 1/5] Michael Phelps...
  [Sample 2/5] Michael Phelps...
  [Sample 3/5] Michael Phelps is nicknamed the 'Baltimore Bullet'....
  [Sample 4/5] Michael Phelps....
  [Sample 5/5] Michael Phelps...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


source_dataset:    triviaqa
correctness:       1.0
energy_score_raw:  1.0000
entropy_score_raw: -0.0000
TBG hidden shape:  (33, 4096)
SLT hidden shape:  (33, 4096)

Test record OK.


## Section 6 — Multi-Dataset Generation Loop

Iterates over each dataset, generates records, and saves per-dataset checkpoints.
Supports resume from checkpoint if interrupted.

In [9]:
import glob as glob_module

def _ckpt_sort_key(path):
    """Extract numeric key from checkpoint filename for correct sorting.
    _final.pkl sorts after all numbered checkpoints."""
    base = os.path.basename(path)
    if '_final.pkl' in base:
        return float('inf')
    m = re.search(r'_(\d+)\.pkl$', base)
    return int(m.group(1)) if m else -1

all_records = []
all_errors  = []

for ds_config in DATASETS:
    tag = ds_config['tag']

    print(f"\n{'='*60}")
    print(f"  Dataset: {tag} — {ds_config['description']}")
    print(f"{'='*60}")

    # ── Load dataset ──────────────────────────────────────────────────────────
    print(f"Loading {ds_config['hf_id']}...")
    if ds_config['config']:
        raw = load_dataset(ds_config['hf_id'], ds_config['config'],
                           split=ds_config['split'], trust_remote_code=True)
    else:
        raw = load_dataset(ds_config['hf_id'],
                           split=ds_config['split'], trust_remote_code=True)

    end_idx = min(ds_config['offset'] + ds_config['num_questions'], len(raw))
    subset = raw.select(range(ds_config['offset'], end_idx))
    actual_n = len(subset)
    print(f"Selected: [{ds_config['offset']}:{end_idx}] = {actual_n} examples")

    # Validate adapter
    ex = subset[0]
    refs = get_refs(ex)
    print(f"  First Q: {ex['question'][:100]}")
    print(f"  Refs:    {refs[:3]}")
    assert len(refs) > 0, f"get_refs() returned empty for {tag}"

    # ── Check for existing checkpoint ─────────────────────────────────────────
    ckpt_prefix = f'checkpoint_{MODEL_SHORT}_mixed_{tag}'
    existing = sorted(
        glob_module.glob(os.path.join(DATA_DIR, f'{ckpt_prefix}_*.pkl')),
        key=_ckpt_sort_key
    )
    ds_records = []
    start_idx = 0

    if existing:
        latest = existing[-1]
        print(f"\nResuming from checkpoint: {os.path.basename(latest)}")
        with open(latest, 'rb') as f:
            ckpt_data = pickle.load(f)
        # Support both old format (list) and new format (dict with next_index)
        if isinstance(ckpt_data, dict) and 'records' in ckpt_data:
            ds_records = ckpt_data['records']
            start_idx = ckpt_data['next_index']
        else:
            ds_records = ckpt_data
            start_idx = len(ds_records)
        print(f"  {len(ds_records)} records loaded, resuming from example index {start_idx}")
    else:
        print(f"\nNo checkpoint found — starting fresh.")

    # ── Generation loop ───────────────────────────────────────────────────────
    errors = []
    for i in tqdm(range(start_idx, actual_n), desc=f'{tag}'):
        example = subset[i]
        try:
            record = generate_record(engine, example, num_samples=NUM_SAMPLES, dataset_tag=tag)
            ds_records.append(record)
        except Exception as e:
            print(f"\n  ERROR [{tag}][{i}]: {e}")
            errors.append({'dataset': tag, 'index': i, 'question': example['question'][:80], 'error': str(e)})
            continue

        # Per-dataset checkpoint — save records + next example index
        if len(ds_records) % CHECKPOINT_EVERY == 0:
            ckpt_path = os.path.join(DATA_DIR, f'{ckpt_prefix}_{len(ds_records)}.pkl')
            with open(ckpt_path, 'wb') as f:
                pickle.dump({'records': ds_records, 'next_index': i + 1}, f)
            print(f"\n  Checkpoint: {os.path.basename(ckpt_path)} ({len(ds_records)} records, next_index={i+1})")

    # Save final checkpoint for this dataset
    final_ckpt = os.path.join(DATA_DIR, f'{ckpt_prefix}_final.pkl')
    with open(final_ckpt, 'wb') as f:
        pickle.dump({'records': ds_records, 'next_index': actual_n}, f)

    print(f"\n{tag}: {len(ds_records)} records collected, {len(errors)} errors")
    all_records.extend(ds_records)
    all_errors.extend(errors)

print(f"\n{'='*60}")
print(f"TOTAL: {len(all_records)} records across {len(DATASETS)} datasets")
print(f"ERRORS: {len(all_errors)}")
if all_errors:
    for e in all_errors[:10]:
        print(f"  [{e['dataset']}][{e['index']}] {e['error'][:80]}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'trivia_qa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.



  Dataset: triviaqa — Factoid trivia questions
Loading trivia_qa...


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Selected: [1000:1400] = 400 examples
  First Q: Which US Olympic swimmer is nicknamed the ‘Baltimore Bullet’?
  Refs:    ['Phelps II', 'Michael Phelps Foundation', 'Michael Fred Phelps']

Resuming from checkpoint: checkpoint_llama-3-1-8b-instruct_mixed_triviaqa_final.pkl
  400 records loaded, resuming from example index 400


...

popqa: 0it [00:00, ?it/s]

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'hotpot_qa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.



popqa: 400 records collected, 0 errors

  Dataset: hotpotqa — Multi-hop reasoning questions
Loading hotpot_qa...
Selected: [0:400] = 400 examples
  First Q: Were Scott Derrickson and Ed Wood of the same nationality?
  Refs:    ['yes']

Resuming from checkpoint: checkpoint_llama-3-1-8b-instruct_mixed_hotpotqa_200.pkl
  200 records loaded, resuming from example index 200


hotpotqa:   0%|          | 0/200 [00:00<?, ?it/s]

  [Sample 1/5] Hulu...
  [Sample 2/5] Hulu....
  [Sample 3/5] Sky One...
  [Sample 4/5] Sky Atlantic...
  [Sample 5/5] Sky Atlantic....
  [Sample 1/5] I do not have any information on a footballer named Chris Williams who played in...
  [Sample 2/5] I'm not aware of any footballer named Chris Williams who played in the National ...
  [Sample 3/5] I do not have any information about a footballer named Chris Williams from the N...
  [Sample 4/5] I don't have information about Chris Williams last playing for a National League...
  [Sample 5/5] I do not have information on Chris Williams....
  [Sample 1/5] I am unable to verify who designed the hotel that held the IFBB professional bod...
  [Sample 2/5] I'm not aware of any information on the hotel that held the IFBB professional bo...
  [Sample 3/5] I am not aware of the information on who designed the hotel that held the IFBB p...
  [Sample 4/5] I am unable to verify who designed the hotel that held the IFBB professional bod...
  [Sample

## Section 7 — Dataset Summary Statistics

In [10]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from collections import Counter

# Filter to records with valid hidden states
valid_records = [r for r in all_records if r['emb_last_tok_before_gen'] is not None]
print(f"Records total:    {len(all_records)}")
print(f"Records with HS:  {len(valid_records)}")
print(f"Dropped (no HS):  {len(all_records) - len(valid_records)}")

# Per-dataset breakdown
ds_counts = Counter(r['source_dataset'] for r in valid_records)
print(f"\n{'Dataset':12s}  {'Count':>6s}  {'Correct':>8s}  {'Energy':>8s}  {'Entropy':>8s}")
print("-" * 50)
for tag in sorted(ds_counts.keys()):
    recs = [r for r in valid_records if r['source_dataset'] == tag]
    corr = np.mean([r['correctness'] for r in recs])
    eng  = np.mean([r['energy_score_raw'] for r in recs])
    ent  = np.mean([r['entropy_score_raw'] for r in recs])
    print(f"{tag:12s}  {len(recs):6d}  {corr:8.3f}  {eng:8.3f}  {ent:8.3f}")

# Aggregate
energy_scores  = np.array([r['energy_score_raw'] for r in valid_records])
entropy_scores = np.array([r['entropy_score_raw'] for r in valid_records])
correctness    = np.array([r['correctness'] for r in valid_records])

print(f"\nAggregate ({len(valid_records)} records):")
print(f"  Energy:  mean={energy_scores.mean():.3f}, std={energy_scores.std():.3f}")
print(f"  Entropy: mean={entropy_scores.mean():.3f}, std={entropy_scores.std():.3f}")
print(f"  Correct: {correctness.mean():.3f} ({int(correctness.sum())}/{len(correctness)})")

# Orientation check
rho_e, p_e = spearmanr(energy_scores, correctness)
rho_h, p_h = spearmanr(entropy_scores, correctness)
print(f"\nOrientation check (Spearman rho):")
print(f"  Energy  rho with correctness: {rho_e:+.3f}  (p={p_e:.2e})  [expected: positive]")
print(f"  Entropy rho with correctness: {rho_h:+.3f}  (p={p_h:.2e})  [expected: negative]")

if rho_e <= 0:
    print("  WARNING: energy orientation looks wrong!")
if rho_h >= 0:
    print("  WARNING: entropy orientation looks wrong!")

Records total:    2000
Records with HS:  1974
Dropped (no HS):  26

Dataset        Count   Correct    Energy   Entropy
--------------------------------------------------
hotpotqa         397     0.207     0.704     0.529
nqopen           396     0.439     0.747     0.478
popqa            400     0.210     0.914     0.147
triviaqa         382     0.762     0.830     0.296
truthfulqa       399     0.070     0.790     0.384

Aggregate (1974 records):
  Energy:  mean=0.797, std=0.285
  Entropy: mean=0.367, std=0.495
  Correct: 0.334 (659/1974)

Orientation check (Spearman rho):
  Energy  rho with correctness: +0.210  (p=4.30e-21)  [expected: positive]
  Entropy rho with correctness: -0.202  (p=1.16e-19)  [expected: negative]


In [11]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

tags_sorted = sorted(ds_counts.keys())

# Energy distribution by dataset
for tag in tags_sorted:
    scores = [r['energy_score_raw'] for r in valid_records if r['source_dataset'] == tag]
    axes[0,0].hist(scores, bins=30, alpha=0.5, label=tag)
axes[0,0].set_xlabel('energy_score_raw')
axes[0,0].set_title('Energy Score Distribution by Dataset')
axes[0,0].legend(fontsize=8)

# Entropy distribution by dataset
for tag in tags_sorted:
    scores = [r['entropy_score_raw'] for r in valid_records if r['source_dataset'] == tag]
    axes[0,1].hist(scores, bins=30, alpha=0.5, label=tag)
axes[0,1].set_xlabel('entropy_score_raw')
axes[0,1].set_title('Entropy Score Distribution by Dataset')
axes[0,1].legend(fontsize=8)

# Correctness rate by dataset
corr_rates = [np.mean([r['correctness'] for r in valid_records if r['source_dataset'] == t]) for t in tags_sorted]
colors = plt.cm.Set2(np.linspace(0, 1, len(tags_sorted)))
axes[1,0].bar(tags_sorted, corr_rates, color=colors)
axes[1,0].set_ylabel('Correctness Rate')
axes[1,0].set_title('Correctness by Dataset')
axes[1,0].tick_params(axis='x', rotation=30)

# Record counts by dataset
sizes = [ds_counts[t] for t in tags_sorted]
axes[1,1].bar(tags_sorted, sizes, color=colors)
axes[1,1].set_ylabel('Record Count')
axes[1,1].set_title('Records per Dataset')
axes[1,1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plot_path = os.path.join(DATA_DIR, 'mixed_dataset_distributions.png')
plt.savefig(plot_path, dpi=120, bbox_inches='tight')
plt.show()
print(f"Plot saved: {plot_path}")

Plot saved: d:\Github Repositories\SemanticEnergy\backend\data\mixed_dataset_distributions.png


C:\Users\diren\AppData\Local\Temp\ipykernel_25520\1918695862.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 8 — Save Final Dataset

In [12]:
# Save final mixed dataset (only records with valid hidden states)
final_records = valid_records

output_path = os.path.join(DATA_DIR, f'probe_dataset_{MODEL_SHORT}_mixed.pkl')
with open(output_path, 'wb') as f:
    pickle.dump(final_records, f)

size_mb = os.path.getsize(output_path) / 1024**2
print(f"Dataset saved: {output_path}")
print(f"Records: {len(final_records)}")
print(f"File size: {size_mb:.1f} MB")

# Per-dataset breakdown
ds_final = Counter(r['source_dataset'] for r in final_records)
print(f"\nPer-dataset breakdown:")
for tag, count in sorted(ds_final.items()):
    print(f"  {tag}: {count}")

# Record fields
print(f"\nRecord fields:")
if final_records:
    for k, v in final_records[0].items():
        if isinstance(v, np.ndarray):
            print(f"  {k}: np.ndarray {v.shape} {v.dtype}")
        elif isinstance(v, dict):
            print(f"  {k}: dict with {len(v)} keys")
        else:
            print(f"  {k}: {type(v).__name__} = {repr(v)[:60]}")

print(f"\nReady for notebook 02_train_se_probes.ipynb")
print(f"Set DATASET_PATH to: {output_path}")

Dataset saved: d:\Github Repositories\SemanticEnergy\backend\data\probe_dataset_llama-3-1-8b-instruct_mixed.pkl
Records: 1974
File size: 2036.8 MB

Per-dataset breakdown:
  hotpotqa: 397
  nqopen: 396
  popqa: 400
  triviaqa: 382
  truthfulqa: 399

Record fields:
  uid: str = 'qb_3329'
  source_dataset: str = 'triviaqa'
  question: str = 'Which US Olympic swimmer is nicknamed the ‘Baltimore Bullet
  main_answer: str = 'Michael Phelps'
  correctness: float = 1.0
  energy_score_raw: float = 1.0
  entropy_score_raw: float = -1.000000082690371e-10
  energy_label: NoneType = None
  entropy_label: NoneType = None
  emb_last_tok_before_gen: np.ndarray (33, 4096) float32
  emb_tok_before_eos: np.ndarray (33, 4096) float32
  logit_feats: dict with 9 keys
  token_ids: list = [26597, 98615]
  num_clusters: int = 1
  cluster_sizes: list = [5]

Ready for notebook 02_train_se_probes.ipynb
Set DATASET_PATH to: d:\Github Repositories\SemanticEnergy\backend\data\probe_dataset_llama-3-1-8b-instruct_mixe

---

# Phase 2 — Probe Training

Train 4 linear probes (SLT/TBG x Energy/Entropy) on the mixed dataset.
Same pipeline as `02_train_se_probes.ipynb` but with stratified splitting by `source_dataset`.

## Section 9 — ProbeDataset + Stratified Split

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
from collections import defaultdict

MODELS_DIR = os.path.join(REPO_ROOT, 'backend', 'models')
FIGS_DIR   = os.path.join(REPO_ROOT, 'notebooks', 'figures')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGS_DIR, exist_ok=True)

np.random.seed(42)

class ProbeDataset:
    """Container for probe training data with hidden states, labels, and logit features."""
    def __init__(self, records, name='train'):
        self.name    = name
        self.N       = len(records)
        self.energy_score_raw  = np.array([r['energy_score_raw']  for r in records])
        self.entropy_score_raw = np.array([r['entropy_score_raw'] for r in records])
        self.correctness       = np.array([r['correctness'] for r in records])
        self.tbg_states = np.stack([r['emb_last_tok_before_gen'] for r in records])
        self.slt_states = np.stack([r['emb_tok_before_eos']      for r in records])
        self.num_layers = self.tbg_states.shape[1]
        self.hidden_dim = self.tbg_states.shape[2]
        self.logit_feats = np.array([
            [r['logit_feats']['mean_chosen_logit'], r['logit_feats']['min_chosen_logit'],
             r['logit_feats']['std_chosen_logit'],  float(r['logit_feats']['answer_len'])]
            for r in records
        ], dtype=np.float64)
        self.logit_feats = np.nan_to_num(self.logit_feats, nan=0.0, posinf=0.0, neginf=0.0)
        self.energy_label  = None
        self.entropy_label = None
        print(f"ProbeDataset '{name}': N={self.N}, layers={self.num_layers}, hidden_dim={self.hidden_dim}")

print("ProbeDataset class defined.")

ProbeDataset class defined.


In [14]:
# ── Stratified 80/10/10 split by source_dataset ──────────────────────────────
# Ensures each dataset is proportionally represented in train/val/test.

all_records_valid = valid_records  # from Section 7

by_dataset = defaultdict(list)
for i, r in enumerate(all_records_valid):
    by_dataset[r['source_dataset']].append(i)

train_idx, val_idx, test_idx = [], [], []
for ds, indices in by_dataset.items():
    perm = np.random.permutation(indices)
    n = len(perm)
    nt = int(0.80 * n)
    nv = int(0.10 * n)
    train_idx.extend(perm[:nt])
    val_idx.extend(perm[nt:nt+nv])
    test_idx.extend(perm[nt+nv:])

np.random.shuffle(train_idx)
np.random.shuffle(val_idx)
np.random.shuffle(test_idx)

train_records = [all_records_valid[i] for i in train_idx]
val_records   = [all_records_valid[i] for i in val_idx]
test_records  = [all_records_valid[i] for i in test_idx]

print(f"Split: train={len(train_records)}, val={len(val_records)}, test={len(test_records)}")

# Verify stratification
for split_name, split_recs in [('train', train_records), ('val', val_records), ('test', test_records)]:
    ds_counts_split = Counter(r['source_dataset'] for r in split_recs)
    print(f"  {split_name}: {dict(sorted(ds_counts_split.items()))}")

D_train = ProbeDataset(train_records, name='train')
D_val   = ProbeDataset(val_records,   name='val')
D_test  = ProbeDataset(test_records,  name='test')

Split: train=1577, val=195, test=202
  train: {'hotpotqa': 317, 'nqopen': 316, 'popqa': 320, 'triviaqa': 305, 'truthfulqa': 319}
  val: {'hotpotqa': 39, 'nqopen': 39, 'popqa': 40, 'triviaqa': 38, 'truthfulqa': 39}
  test: {'hotpotqa': 41, 'nqopen': 41, 'popqa': 40, 'triviaqa': 39, 'truthfulqa': 41}
ProbeDataset 'train': N=1577, layers=33, hidden_dim=4096
ProbeDataset 'val': N=195, layers=33, hidden_dim=4096
ProbeDataset 'test': N=202, layers=33, hidden_dim=4096


## Section 10 — Binarization + Training Functions

In [15]:
# ── Binarization (MSE-optimal thresholding on train set) ─────────────────────

def find_best_threshold(scores, label='scores'):
    """SEP-style: find threshold minimizing within-group MSE."""
    best_thresh, best_mse = None, float('inf')
    for pct in np.arange(10, 91, 1):
        thresh = np.percentile(scores, pct)
        g0, g1 = scores[scores < thresh], scores[scores >= thresh]
        if len(g0) == 0 or len(g1) == 0:
            continue
        mse = (len(g0) * g0.var() + len(g1) * g1.var()) / len(scores)
        if mse < best_mse:
            best_mse, best_thresh = mse, thresh
    return best_thresh

def binarize(scores, threshold):
    return (scores >= threshold).astype(int)

T_energy  = find_best_threshold(D_train.energy_score_raw,  label='energy')
T_entropy = find_best_threshold(D_train.entropy_score_raw, label='entropy')

print(f"Energy threshold  T_energy  = {T_energy:.4f}")
print(f"Entropy threshold T_entropy = {T_entropy:.4f}")

for D in [D_train, D_val, D_test]:
    D.energy_label  = binarize(D.energy_score_raw,  T_energy)
    D.entropy_label = binarize(D.entropy_score_raw, T_entropy)

for D in [D_train, D_val, D_test]:
    print(f"  {D.name}: energy 1s={D.energy_label.sum()}/{D.N} ({D.energy_label.mean():.1%}), "
          f"entropy 1s={D.entropy_label.sum()}/{D.N} ({D.entropy_label.mean():.1%})")

# ── Training utilities ───────────────────────────────────────────────────────

def get_layer_X(D, layer_idx, token='slt'):
    states = D.slt_states if token == 'slt' else D.tbg_states
    return states[:, layer_idx, :]

def get_range_X(D, layer_range, token='slt'):
    l_start, l_end = layer_range
    states = D.slt_states if token == 'slt' else D.tbg_states
    return states[:, l_start:l_end, :].reshape(D.N, -1)

def clean_X(X):
    return np.nan_to_num(np.array(X, dtype=np.float64), nan=0.0, posinf=0.0, neginf=0.0)

def sklearn_train_eval(X_train, y_train, X_val, y_val, scale=True):
    X_train, X_val = clean_X(X_train), clean_X(X_val)
    if scale:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val   = scaler.transform(X_val)
    else:
        scaler = None
    probe = LogisticRegression(max_iter=1000, C=1.0)
    probe.fit(X_train, y_train)
    y_score = probe.predict_proba(X_val)[:, 1]
    auroc = roc_auc_score(y_val, y_score)
    return probe, scaler, auroc

def bootstrap_auroc(probe, scaler, X_test, y_test, n_boot=1000, ci=0.95):
    if scaler is not None:
        X_test = scaler.transform(clean_X(X_test))
    y_score = probe.predict_proba(X_test)[:, 1]
    base_auroc = roc_auc_score(y_test, y_score)
    boot_aurocs = []
    rng = np.random.default_rng(0)
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_test), len(y_test))
        if len(np.unique(y_test[idx])) < 2:
            continue
        boot_aurocs.append(roc_auc_score(y_test[idx], y_score[idx]))
    alpha = (1 - ci) / 2
    return {'mean': base_auroc, 'lo': np.percentile(boot_aurocs, alpha*100), 'hi': np.percentile(boot_aurocs, (1-alpha)*100)}

def decide_layer_range(auroc_list, window_sizes=[4, 8, 16]):
    aucs = np.array(auroc_list)
    best_mean, best_range = -np.inf, (0, 4)
    for window in window_sizes:
        for start in range(len(aucs) - window + 1):
            end = start + window
            mean_auc = aucs[start:end].mean()
            if mean_auc > best_mean:
                best_mean, best_range = mean_auc, (start, end)
    return best_mean, best_range

def save_fig(name):
    path = os.path.join(FIGS_DIR, f'{name}.png')
    plt.savefig(path, dpi=120, bbox_inches='tight')
    print(f"Saved: {path}")

print("Training functions defined.")

Energy threshold  T_energy  = 0.7718
Entropy threshold T_entropy = 0.6730
  train: energy 1s=1104/1577 (70.0%), entropy 1s=451/1577 (28.6%)
  val: energy 1s=139/195 (71.3%), entropy 1s=51/195 (26.2%)
  test: energy 1s=144/202 (71.3%), entropy 1s=52/202 (25.7%)
Training functions defined.


## Section 11 — Per-Layer AUROC Sweep

In [16]:
# Energy probe layer sweep
energy_slt_aurocs, energy_tbg_aurocs = [], []

for layer in tqdm(range(D_train.num_layers), desc='Energy layer sweep'):
    X_train_slt = get_layer_X(D_train, layer, 'slt')
    X_val_slt   = get_layer_X(D_val,   layer, 'slt')
    _, _, auroc_slt = sklearn_train_eval(X_train_slt, D_train.energy_label, X_val_slt, D_val.energy_label)
    energy_slt_aurocs.append(auroc_slt)
    
    X_train_tbg = get_layer_X(D_train, layer, 'tbg')
    X_val_tbg   = get_layer_X(D_val,   layer, 'tbg')
    _, _, auroc_tbg = sklearn_train_eval(X_train_tbg, D_train.energy_label, X_val_tbg, D_val.energy_label)
    energy_tbg_aurocs.append(auroc_tbg)

# Entropy probe layer sweep
entropy_slt_aurocs, entropy_tbg_aurocs = [], []

for layer in tqdm(range(D_train.num_layers), desc='Entropy layer sweep'):
    X_train_slt = get_layer_X(D_train, layer, 'slt')
    X_val_slt   = get_layer_X(D_val,   layer, 'slt')
    _, _, auroc_slt = sklearn_train_eval(X_train_slt, D_train.entropy_label, X_val_slt, D_val.entropy_label)
    entropy_slt_aurocs.append(auroc_slt)
    
    X_train_tbg = get_layer_X(D_train, layer, 'tbg')
    X_val_tbg   = get_layer_X(D_val,   layer, 'tbg')
    _, _, auroc_tbg = sklearn_train_eval(X_train_tbg, D_train.entropy_label, X_val_tbg, D_val.entropy_label)
    entropy_tbg_aurocs.append(auroc_tbg)

print("Layer sweep complete.")

Energy layer sweep:   0%|          | 0/33 [00:00<?, ?it/s]

Entropy layer sweep:   0%|          | 0/33 [00:00<?, ?it/s]

Layer sweep complete.


In [17]:
# Layer sweep visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
layers = range(D_train.num_layers)

axes[0].plot(layers, energy_slt_aurocs, 'o-', label='SLT', markersize=4)
axes[0].plot(layers, energy_tbg_aurocs, 's-', label='TBG', markersize=4)
axes[0].set_xlabel('Layer'); axes[0].set_ylabel('AUROC')
axes[0].set_title('Energy Probe — Per-Layer AUROC (Mixed Dataset)')
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

axes[1].plot(layers, entropy_slt_aurocs, 'o-', label='SLT', markersize=4)
axes[1].plot(layers, entropy_tbg_aurocs, 's-', label='TBG', markersize=4)
axes[1].set_xlabel('Layer'); axes[1].set_ylabel('AUROC')
axes[1].set_title('Entropy Probe — Per-Layer AUROC (Mixed Dataset)')
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
save_fig('mixed_layer_sweep')
plt.show()

Saved: d:\Github Repositories\SemanticEnergy\notebooks\figures\mixed_layer_sweep.png


C:\Users\diren\AppData\Local\Temp\ipykernel_25520\1391466006.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 12 — Best Layer Range + Final Training + Evaluation

In [22]:
# ── Best layer ranges ─────────────────────────────────────────────────────────
best_energy_slt_mean,  best_energy_slt_range  = decide_layer_range(energy_slt_aurocs)
best_energy_tbg_mean,  best_energy_tbg_range  = decide_layer_range(energy_tbg_aurocs)
best_entropy_slt_mean, best_entropy_slt_range = decide_layer_range(entropy_slt_aurocs)
best_entropy_tbg_mean, best_entropy_tbg_range = decide_layer_range(entropy_tbg_aurocs)

print("Best layer ranges (contiguous window with highest mean validation AUROC):")
print(f"  Energy  SLT: layers {best_energy_slt_range}   mean AUROC = {best_energy_slt_mean:.4f}")
print(f"  Energy  TBG: layers {best_energy_tbg_range}   mean AUROC = {best_energy_tbg_mean:.4f}")
print(f"  Entropy SLT: layers {best_entropy_slt_range}  mean AUROC = {best_entropy_slt_mean:.4f}")
print(f"  Entropy TBG: layers {best_entropy_tbg_range}  mean AUROC = {best_entropy_tbg_mean:.4f}")

Best layer ranges (contiguous window with highest mean validation AUROC):
  Energy  SLT: layers (16, 20)   mean AUROC = 0.7917
  Energy  TBG: layers (13, 17)   mean AUROC = 0.7029
  Entropy SLT: layers (17, 21)  mean AUROC = 0.7833
  Entropy TBG: layers (13, 17)  mean AUROC = 0.7001


In [19]:
# ── Train final probes + evaluate on test set ─────────────────────────────────

trained_probes = {}

configs = [
    ('slt_energy',  'energy_label',  'slt', best_energy_slt_range),
    ('tbg_energy',  'energy_label',  'tbg', best_energy_tbg_range),
    ('slt_entropy', 'entropy_label', 'slt', best_entropy_slt_range),
    ('tbg_entropy', 'entropy_label', 'tbg', best_entropy_tbg_range),
]

print("Training final probes on train set, evaluating on test set:\n")
print(f"{'Probe':<15} {'AUROC':>8} {'95% CI':>20} {'Layer Range':>14}")
print("-" * 60)

for probe_name, label_key, token, layer_range in configs:
    X_train = get_range_X(D_train, layer_range, token)
    X_val   = get_range_X(D_val,   layer_range, token)
    X_test  = get_range_X(D_test,  layer_range, token)
    y_train = getattr(D_train, label_key)
    y_val   = getattr(D_val,   label_key)
    y_test  = getattr(D_test,  label_key)
    
    probe, scaler, val_auroc = sklearn_train_eval(X_train, y_train, X_val, y_val)
    ci = bootstrap_auroc(probe, scaler, X_test, y_test)
    
    trained_probes[probe_name] = {'probe': probe, 'scaler': scaler, 'layer_range': layer_range, 'token': token}
    print(f"{probe_name:<15} {ci['mean']:>8.4f} [{ci['lo']:.4f}, {ci['hi']:.4f}]  {str(layer_range):>14}")

Training final probes on train set, evaluating on test set:

Probe              AUROC               95% CI    Layer Range
------------------------------------------------------------
slt_energy        0.7614 [0.6851, 0.8321]        (16, 20)
tbg_energy        0.6797 [0.5980, 0.7562]        (13, 17)
slt_entropy       0.7682 [0.6884, 0.8385]        (17, 21)
tbg_entropy       0.6609 [0.5812, 0.7369]        (13, 17)


In [ ]:
# ── Hallucination detection AUROC (correctness-based) ─────────────────────────
# Probes predict teacher labels, but we ultimately care about correctness.
# Include teacher signals as baselines for comparison.

y_hallucination = (D_test.correctness == 0).astype(int)  # 1 = hallucinated

print(f"Test set hallucination rate: {y_hallucination.mean():.1%} ({y_hallucination.sum()}/{len(y_hallucination)})")
print()

# Per-dataset breakdown on test set
test_ds_tags = [test_records[i]['source_dataset'] for i in range(len(test_records))]
print(f"{'System':<20} {'Overall':>8}", end='')
test_tags_unique = sorted(set(test_ds_tags))
for t in test_tags_unique:
    print(f" {t:>10}", end='')
print()
print("-" * (30 + 11 * len(test_tags_unique)))

# ── Teacher signal baselines ──────────────────────────────────────────────────
energy_raw  = np.array([test_records[i]['energy_score_raw'] for i in range(len(test_records))])
entropy_raw = np.array([test_records[i]['entropy_score_raw'] for i in range(len(test_records))])

for teacher_name, teacher_scores in [('energy_teacher', 1.0 - energy_raw),
                                      ('entropy_teacher', entropy_raw)]:
    overall_auroc = roc_auc_score(y_hallucination, teacher_scores)
    print(f"{teacher_name:<20} {overall_auroc:>8.4f}", end='')
    for t in test_tags_unique:
        mask = np.array([tag == t for tag in test_ds_tags])
        if mask.sum() < 5 or len(np.unique(y_hallucination[mask])) < 2:
            print(f" {'N/A':>10}", end='')
        else:
            ds_auroc = roc_auc_score(y_hallucination[mask], teacher_scores[mask])
            print(f" {ds_auroc:>10.4f}", end='')
    print()

print("-" * (30 + 11 * len(test_tags_unique)))

# ── Probe scores ──────────────────────────────────────────────────────────────
for probe_name, label_key, token, layer_range in configs:
    # Overall
    X_test = get_range_X(D_test, layer_range, token)
    if trained_probes[probe_name]['scaler'] is not None:
        X_test_s = trained_probes[probe_name]['scaler'].transform(clean_X(X_test))
    else:
        X_test_s = clean_X(X_test)
    
    # Energy probes: high score = confident -> invert for hallucination risk
    # Entropy probes: high score = uncertain -> direct hallucination risk
    scores = trained_probes[probe_name]['probe'].predict_proba(X_test_s)[:, 1]
    if 'energy' in probe_name:
        risk_scores = 1.0 - scores
    else:
        risk_scores = scores
    
    overall_auroc = roc_auc_score(y_hallucination, risk_scores)
    print(f"{probe_name:<20} {overall_auroc:>8.4f}", end='')
    
    # Per-dataset AUROC
    for t in test_tags_unique:
        mask = np.array([tag == t for tag in test_ds_tags])
        if mask.sum() < 5 or len(np.unique(y_hallucination[mask])) < 2:
            print(f" {'N/A':>10}", end='')
        else:
            ds_auroc = roc_auc_score(y_hallucination[mask], risk_scores[mask])
            print(f" {ds_auroc:>10.4f}", end='')
    print()

Test set hallucination rate: 68.3% (138/202)

System                Overall   hotpotqa     nqopen      popqa   triviaqa truthfulqa
-------------------------------------------------------------------------------------
energy_teacher         0.6025     0.7500     0.5335     0.5233     0.7862     0.5263
entropy_teacher        0.5917     0.7095     0.4964     0.5251     0.7655     0.5351
-------------------------------------------------------------------------------------
slt_energy             0.5452     0.6419     0.5574     0.3513     0.6759     0.6579
tbg_energy             0.5331     0.6554     0.3660     0.4194     0.7345     0.7368
slt_entropy            0.5580     0.6284     0.5478     0.3405     0.6552     0.7544
tbg_entropy            0.5289     0.6216     0.3541     0.4158     0.7310     0.6667


: 

## Section 13 — Save Probe Bundle

In [21]:
probe_bundle = {
    'model_id':                MODEL_ID,
    'dataset':                 'mixed',
    'datasets_used':           [d['tag'] for d in DATASETS],
    'num_per_dataset':         dict(Counter(r['source_dataset'] for r in train_records)),
    'num_train_records':       D_train.N,
    
    # Binarization thresholds (computed on train split)
    'energy_threshold':        T_energy,
    'entropy_threshold':       T_entropy,
    
    # Best layer ranges
    'best_energy_slt_range':   best_energy_slt_range,
    'best_energy_tbg_range':   best_energy_tbg_range,
    'best_entropy_slt_range':  best_entropy_slt_range,
    'best_entropy_tbg_range':  best_entropy_tbg_range,
    
    # Trained probes and scalers
    'slt_energy_probe':        trained_probes['slt_energy']['probe'],
    'slt_energy_scaler':       trained_probes['slt_energy']['scaler'],
    'tbg_energy_probe':        trained_probes['tbg_energy']['probe'],
    'tbg_energy_scaler':       trained_probes['tbg_energy']['scaler'],
    'slt_entropy_probe':       trained_probes['slt_entropy']['probe'],
    'slt_entropy_scaler':      trained_probes['slt_entropy']['scaler'],
    'tbg_entropy_probe':       trained_probes['tbg_entropy']['probe'],
    'tbg_entropy_scaler':      trained_probes['tbg_entropy']['scaler'],
    
    # Per-layer AUROC results (for inspection)
    'layer_auroc_table': {
        'energy_slt':  energy_slt_aurocs,
        'energy_tbg':  energy_tbg_aurocs,
        'entropy_slt': entropy_slt_aurocs,
        'entropy_tbg': entropy_tbg_aurocs,
    },
}

output_path = os.path.join(MODELS_DIR, f'probes_{MODEL_SHORT}_mixed.pkl')
with open(output_path, 'wb') as f:
    pickle.dump(probe_bundle, f)

size_kb = os.path.getsize(output_path) / 1024
print(f"Probe bundle saved: {output_path}")
print(f"File size: {size_kb:.0f} KB")
print()
print("Contents:")
for k, v in probe_bundle.items():
    if hasattr(v, '__class__') and 'LogisticRegression' in type(v).__name__:
        print(f"  {k}: LogisticRegression")
    elif isinstance(v, list):
        print(f"  {k}: list[{len(v)}]")
    elif isinstance(v, dict):
        print(f"  {k}: dict with {len(v)} keys")
    elif isinstance(v, tuple):
        print(f"  {k}: {v}")
    else:
        print(f"  {k}: {type(v).__name__} = {repr(v)[:60]}")

print(f"\nTo use these probes, update backend/app.py PROBE_BUNDLES:")
print(f'  "{MODEL_ID}": "probes_{MODEL_SHORT}_mixed.pkl"')

Probe bundle saved: d:\Github Repositories\SemanticEnergy\backend\models\probes_llama-3-1-8b-instruct_mixed.pkl
File size: 2052 KB

Contents:
  model_id: str = 'meta-llama/Llama-3.1-8B-Instruct'
  dataset: str = 'mixed'
  datasets_used: list[5]
  num_per_dataset: dict with 5 keys
  num_train_records: int = 1577
  energy_threshold: float64 = np.float64(0.7717767568671889)
  entropy_threshold: float64 = np.float64(0.6730116668092565)
  best_energy_slt_range: (16, 20)
  best_energy_tbg_range: (13, 17)
  best_entropy_slt_range: (17, 21)
  best_entropy_tbg_range: (13, 17)
  slt_energy_probe: LogisticRegression
  slt_energy_scaler: StandardScaler = StandardScaler()
  tbg_energy_probe: LogisticRegression
  tbg_energy_scaler: StandardScaler = StandardScaler()
  slt_entropy_probe: LogisticRegression
  slt_entropy_scaler: StandardScaler = StandardScaler()
  tbg_entropy_probe: LogisticRegression
  tbg_entropy_scaler: StandardScaler = StandardScaler()
  layer_auroc_table: dict with 4 keys

To use 